# Chapter 33 bridge: BPTT and attention against autograd

Chapter 33 derives backpropagation through time by hand, and Step 6 adds attention. Both backward passes are checked here against autograd, and so is Step 2's state Jacobian: its two factors per step do not commute, and only one order is the derivative.

Nothing here is reimplemented. The chapter's own file is executed, its own weights and data are handed to PyTorch, and the chapter's hand-derived gradients are compared against autograd. Tolerances are stated per check and are **relative** to the size of the quantity being compared, because an absolute threshold means nothing without a scale.

Run order: top to bottom, from a fresh kernel. Requires `requirements-bridges.txt` on top of the book's own `requirements.txt`.

In [1]:
import os, sys, numpy as np, torch
torch.set_default_dtype(torch.float64)          # match NumPy's float64 exactly
CH = os.path.join("..", "code", "ch33")
os.chdir(CH) if os.path.basename(os.getcwd()) != "ch33" else None
def run(name):
    exec(open(name, encoding="utf-8").read(), globals())
run("_lib.py")
print("chapter:", os.path.basename(os.getcwd()), "| torch", torch.__version__, "| numpy", np.__version__)


chapter: ch33 | torch 2.14.0 | numpy 2.4.4


In [2]:
def report(name, ours, theirs, tol=1e-9):
    a = np.asarray(ours, dtype=float); b = np.asarray(theirs, dtype=float)
    denom = max(np.abs(b).max(), 1e-300)
    absd = np.abs(a - b).max(); rel = absd / denom
    ok = rel <= tol
    RESULTS.append(dict(check=name, max_abs=float(absd), max_rel=float(rel),
                        scale=float(denom), tol=tol, passed=bool(ok)))
    print(f"{'PASS' if ok else 'FAIL'}  {name:52s} max|diff| {absd:.3e}   "
          f"relative {rel:.2e}   (tolerance {tol:g})")
    return ok
RESULTS = []
MEASUREMENTS = []      # reported, never asserted: these have no single right answer


In [3]:
run("c1.py"); run("c2.py"); run("c3.py")

 timestep   row (input)  |hidden state|
        0          (8,)          2.1026
        1          (8,)          2.2817
        2          (8,)          2.0675
        3          (8,)          2.0757
        4          (8,)          2.1791
        5          (8,)          2.1717
        6          (8,)          2.4250
        7          (8,)          2.6378

after all 8 rows, one hidden state of size 16 summarizes
the entire image: [-0.579  0.809  0.798 -0.067 -0.242 -0.004 -0.738  0.509 -0.034  0.937
 -0.882  0.746 -0.898 -0.808  0.779 -0.435]
 sequence length gradient norm, step 1 to last
               5                      5.15e-01
              10                      1.48e-01
              20                      1.02e-03
              40                      3.97e-06
              80                      5.66e-15
        target    analytic   numerical   match
Wx(4, 8)    4.875486    4.875486    True
Wx(3, 4)    3.821714    3.821714    True
Wh(12, 8)   -5.155417   -5.155417    T

### Backpropagation through time

In [4]:
rng = np.random.default_rng(4)
n, T, D_in, D_hid = 16, 7, 3, 12
Xs = rng.normal(0, 1, (n, T, D_in))
Wx = rng.normal(0, 0.4, (D_in, D_hid)); Wh = rng.normal(0, 0.25, (D_hid, D_hid)); bh = np.zeros(D_hid)
hs = rnn_forward(Xs, Wx, Wh, bh)
dh_last = rng.normal(0, 1, (n, D_hid))
dWx, dWh, dbh = rnn_backward(dh_last, Xs, hs, Wx, Wh)

tWx = torch.tensor(Wx, requires_grad=True); tWh = torch.tensor(Wh, requires_grad=True)
tbh = torch.tensor(bh, requires_grad=True); tX = torch.tensor(Xs)
h = torch.zeros(n, D_hid, dtype=torch.float64)
for t in range(T):
    h = torch.tanh(tX[:, t] @ tWx + h @ tWh + tbh)
h.backward(torch.tensor(dh_last))
report("BPTT: dWx", dWx, tWx.grad.numpy())
report("BPTT: dWh", dWh, tWh.grad.numpy())
report("BPTT: dbh", dbh, tbh.grad.numpy())

PASS  BPTT: dWx                                            max|diff| 3.553e-15   relative 3.98e-16   (tolerance 1e-09)
PASS  BPTT: dWh                                            max|diff| 2.220e-15   relative 4.01e-16   (tolerance 1e-09)
PASS  BPTT: dbh                                            max|diff| 2.220e-15   relative 3.53e-16   (tolerance 1e-09)


np.True_

### Step 2's state Jacobian
Step 2 multiplies one factor per step to get the gradient of the last hidden state with respect to the first, and prints its norm. The three checks above cover the *parameter* gradients of `rnn_backward`; they say nothing about that product. This reads `run_and_track_grad` out of `c2.py` at run time, keeps its arithmetic and its random draws, returns the matrix instead of its norm, and compares it with autograd's Jacobian of the same recurrence in the same convention: row *i* is how the last state moves when element *i* of the first state moves. A product with its two factors in the other order does not pass this check.

In [5]:
import re
src = open("c2.py", encoding="utf-8").read()
m = re.search(r"^def run_and_track_grad\(.*?(?=^\S|\Z)", src, re.S | re.M)
body = m.group(0).rstrip().replace("return np.linalg.norm(grad)", "return grad")
g = dict(globals()); g["D_hid"] = 16          # c2.py's own width; the BPTT cell above used 12
exec(body, g)
# replay the chapter's random stream exactly: the weights once, then the inputs for 5, 10 and
# 20 steps in that order, so the norms printed here are the ones the book prints
g["r2"] = np.random.default_rng(33); g["Wh_decay"] = g["r2"].normal(0, 0.3, (16, 16))
r_probe = np.random.default_rng(33); r_probe.normal(0, 0.3, (16, 16))
for n_steps in (5, 10, 20):
    J_chapter = g["run_and_track_grad"](n_steps, g["Wh_decay"])
    xs = torch.tensor(r_probe.normal(size=(n_steps, 16))); W = torch.tensor(g["Wh_decay"])
    def last_state(h0):
        h = h0
        for x in xs:
            h = torch.tanh(x + h @ W)
        return h
    J_auto = torch.autograd.functional.jacobian(last_state, torch.zeros(16, dtype=torch.float64)).T
    report(f"Step 2 state Jacobian, {n_steps} steps: d(last state)/d(first state)", J_chapter, J_auto.numpy())
    print(f"        norm {np.linalg.norm(J_chapter):.10f}")

PASS  Step 2 state Jacobian, 5 steps: d(last state)/d(first state) max|diff| 6.939e-17   relative 3.52e-16   (tolerance 1e-09)
        norm 0.5151874816


PASS  Step 2 state Jacobian, 10 steps: d(last state)/d(first state) max|diff| 3.123e-17   relative 9.28e-16   (tolerance 1e-09)
        norm 0.1483736332
PASS  Step 2 state Jacobian, 20 steps: d(last state)/d(first state) max|diff| 3.253e-19   relative 1.10e-15   (tolerance 1e-09)
        norm 0.0010196065


### Step 6's attention backward pass
This runs **the chapter's own `train_recall_attention`, read from `c6.py` at run time**, for a single step, and compares the weights it produces against one step of the same update in PyTorch. It therefore checks the file, not a copy of it: change the chapter's gradient and this check moves with it.

In [6]:
run("c5.py"); run("c6.py")

 sequence length   recall accuracy
               2            1.0000
               5            1.0000


              10            0.6600


              20            0.3050


              40            0.3250

chance accuracy on three classes: 0.333
 sequence length   plain RNN  with attention
               2      1.0000          1.0000
               5      1.0000          1.0000


              10      0.6600          1.0000


              20      0.3050          1.0000


              40      0.3250          0.9950

attention weights on the length-40 task, averaged over the test set:
[0.084 0.036 0.026 0.025 0.024 0.025 0.024 0.024 0.023 0.023 0.023 0.023
 0.023 0.023 0.023 0.023 0.023 0.023 0.023 0.023 0.023 0.023 0.023 0.023
 0.023 0.023 0.023 0.023 0.023 0.023 0.023 0.022 0.023 0.023 0.023 0.023
 0.023 0.022 0.023 0.022]


In [7]:

import re
def one_step(filename, funcname, *a, **kw):
    """Run ONE training step of the chapter's own function and hand back its weights.

    The function's source is READ OUT OF THE CHAPTER FILE at run time, not retyped here and
    not taken from an already-imported object. Edit the chapter's gradient and this check
    moves with it, which is the whole point of a bridge. Only the final `return` is rewritten,
    so the locals -- the updated weights -- come back.
    """
    src = open(filename, encoding='utf-8').read()
    m = re.search(r'^def ' + funcname + r'\(.*?(?=^\S|\Z)', src, re.S | re.M)
    if not m:
        raise SystemExit(funcname + ' not found in ' + filename)
    lines = m.group(0).rstrip().split(chr(10))
    lines[0] = re.sub(r'^def ' + funcname, 'def _instrumented', lines[0])
    for i in range(len(lines) - 1, -1, -1):
        if lines[i].lstrip().startswith('return '):
            pad = lines[i][:len(lines[i]) - len(lines[i].lstrip())]
            lines[i] = pad + 'return locals()'
            break
    g = dict(globals())
    exec(chr(10).join(lines), g)
    print('checking', funcname, 'as read from', filename)
    return g['_instrumented'](*a, **kw)


In [8]:
SEQ, SEED, D_hid, ETA = 9, 3, 12, 0.5
st = one_step("c6.py", "train_recall_attention", SEQ, SEED, epochs=1, D_hid=D_hid)
after = {k: st[k] for k in ("Wx", "Wh", "bh", "q", "Wo", "bo")}

# reproduce the chapter's own initialisation and data, independently
Xs, ys = make_recall_task(600, SEQ, SEED); Y = np.eye(3)[ys]
rr = np.random.default_rng(SEED)
Wx0 = rr.normal(0, np.sqrt(1/3), (3, D_hid)); Wh0 = rr.normal(0, np.sqrt(1/D_hid), (D_hid, D_hid))
bh0 = np.zeros(D_hid); q0 = rr.normal(0, np.sqrt(1/D_hid), D_hid)
Wo0 = rr.normal(0, np.sqrt(1/D_hid), (D_hid, 3)); bo0 = np.zeros(3)

T = {n: torch.tensor(v, requires_grad=True) for n, v in
     dict(Wx=Wx0, Wh=Wh0, bh=bh0, q=q0, Wo=Wo0, bo=bo0).items()}
tX = torch.tensor(Xs)
h = torch.zeros(len(Xs), D_hid, dtype=torch.float64); H = []
for t in range(SEQ):
    h = torch.tanh(tX[:, t] @ T["Wx"] + h @ T["Wh"] + T["bh"]); H.append(h)
H = torch.stack(H, 1)
w = torch.softmax(H @ T["q"], dim=1)
ctx = torch.einsum('nt,nth->nh', w, H)
tp = torch.softmax(ctx @ T["Wo"] + T["bo"], dim=1)
(-(torch.tensor(Y) * torch.log(tp + 1e-12)).sum() / len(Xs)).backward()

for n in ("Wo", "bo", "q", "Wx", "Wh", "bh"):
    torch_after = (T[n] - ETA * T[n].grad).detach().numpy()
    report(f"one step of train_recall_attention(): {n} after the update", after[n], torch_after)

checking train_recall_attention as read from c6.py
PASS  one step of train_recall_attention(): Wo after the update max|diff| 8.099e-14   relative 1.32e-13   (tolerance 1e-09)
PASS  one step of train_recall_attention(): bo after the update max|diff| 1.840e-13   relative 5.66e-12   (tolerance 1e-09)
PASS  one step of train_recall_attention(): q after the update max|diff| 2.753e-14   relative 4.42e-14   (tolerance 1e-09)
PASS  one step of train_recall_attention(): Wx after the update max|diff| 4.025e-14   relative 2.10e-14   (tolerance 1e-09)
PASS  one step of train_recall_attention(): Wh after the update max|diff| 9.190e-14   relative 1.12e-13   (tolerance 1e-09)
PASS  one step of train_recall_attention(): bh after the update max|diff| 1.341e-13   relative 4.63e-12   (tolerance 1e-09)


In [9]:
import json
n_pass = sum(1 for r in RESULTS if r["passed"])
print(f"\n{n_pass} of {len(RESULTS)} ASSERTED checks passed")
if MEASUREMENTS:
    print(f"{len(MEASUREMENTS)} reported measurement(s), not asserted:")
    for m in MEASUREMENTS: print("   ", m)
print(json.dumps(dict(checks=RESULTS, measurements=MEASUREMENTS), indent=1))
assert n_pass == len(RESULTS), "a gradient check failed"



12 of 12 ASSERTED checks passed
{
 "checks": [
  {
   "check": "BPTT: dWx",
   "max_abs": 3.552713678800501e-15,
   "max_rel": 3.976617480818073e-16,
   "scale": 8.934009106829238,
   "tol": 1e-09,
   "passed": true
  },
  {
   "check": "BPTT: dWh",
   "max_abs": 2.220446049250313e-15,
   "max_rel": 4.0081085789532166e-16,
   "scale": 5.539884974449018,
   "tol": 1e-09,
   "passed": true
  },
  {
   "check": "BPTT: dbh",
   "max_abs": 2.220446049250313e-15,
   "max_rel": 3.533121007992622e-16,
   "scale": 6.284658929674989,
   "tol": 1e-09,
   "passed": true
  },
  {
   "check": "Step 2 state Jacobian, 5 steps: d(last state)/d(first state)",
   "max_abs": 6.938893903907228e-17,
   "max_rel": 3.5235435421973896e-16,
   "scale": 0.19692942121498297,
   "tol": 1e-09,
   "passed": true
  },
  {
   "check": "Step 2 state Jacobian, 10 steps: d(last state)/d(first state)",
   "max_abs": 3.122502256758253e-17,
   "max_rel": 9.28434479943835e-16,
   "scale": 0.033631907519711526,
   "tol": 1e-